In [523]:
import os
import torch
import numpy as np
from nelora_utils import config_nelora as nelora_config
from models.loraseek.loraseek_model.model_LoRaSeek import LoRaSeekNet
from nelora_utils.utils import spec_to_network_input
from sean_utils.LoRa import LoRa

def print_opts(opts):
    """Prints the values of all configuration arguments."""
    print('=' * 50)
    print('LoRaSeek Configuration:')
    print('=' * 50)
    for key, val in vars(opts).items():
        print(f'{key}: {val}')
    print('=' * 50)

def LoRaSeek_Init(loraseek_path, sf=7):
    parser = nelora_config.create_parser()
    # Pass [] so parse_args doesn't parse Jupyter kernel command-line arguments
    opts = parser.parse_args([])
    opts.sf = sf
    opts.n_classes = 2 ** opts.sf
    opts.stft_nfft = opts.n_classes * opts.fs // opts.bw
    opts.stft_window = opts.n_classes // 2
    opts.stft_overlap = opts.stft_window // 2
    opts.conv_dim_lstm = opts.n_classes * opts.fs // opts.bw
    opts.freq_size = opts.n_classes
    opts.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # LoRaSeek-specific architecture hyperparameters (from v9_checkpoints)
    opts.base_channel = 16
    opts.channel_attention_reduction = 8
    opts.num_of_transformers = 4
    opts.num_of_heads = 2

    print_opts(opts)
    print(f'Using device: {opts.device}')

    # Initialize and load LoRaSeekNet
    loraseek = LoRaSeekNet(opts)
    loraseek_weights = torch.load(loraseek_path, map_location=opts.device, weights_only=False)
    loraseek.load_state_dict(loraseek_weights, strict=True)
    loraseek.to(opts.device)
    loraseek.eval()

    print('LoRaSeekNet model successfully loaded and set to eval mode.')
    return loraseek, opts

# --------------------------------------------------------------------------
# Helper: Denoise complex IQ symbol using LoRaSeek + ISTFT
# --------------------------------------------------------------------------
def denoise_with_loraseek_istft(sig_iq, loraseek, opts, window):
    """
    1. Computes STFT of noisy complex IQ samples
    2. Preprocesses spectrogram into network input [1, 2, 128, 32]
    3. Runs forward pass through LoRaSeekNet to obtain denoised spectrogram
    4. Maps back to full frequency spectrum and runs ISTFT to reconstruct clean IQ signal
    """
    t_in = torch.from_numpy(sig_iq).unsqueeze(0).to(dtype=torch.complex64, device=opts.device)
    raw_spec = torch.stft(
        input=t_in,
        n_fft=opts.stft_nfft,
        hop_length=opts.stft_overlap,
        win_length=opts.stft_window,
        window=window,
        pad_mode='constant',
        return_complex=True
    )
    
    # Trim central bandwidth and normalize
    freq_size = opts.freq_size
    trim_size = freq_size // 2
    y = torch.cat((raw_spec[:, -trim_size:, :], raw_spec[:, 0:trim_size, :]), 1)
    y_abs = torch.abs(y)
    y_abs_max = torch.amax(y_abs, dim=(1, 2), keepdim=True)
    y_norm = torch.div(y, y_abs_max)
    
    # Format into [1, 2, 128, 32]
    net_in = torch.view_as_real(y_norm).permute(0, 3, 1, 2)
    net_in_32 = net_in[:, :, :, :32]
    
    with torch.no_grad():
        denoised_net = loraseek(net_in_32)  # [1, 2, 128, 32]
        
    # Convert back to complex and denormalize
    denoised_complex_128 = torch.view_as_complex(denoised_net.permute(0, 2, 3, 1).contiguous())
    denoised_complex_128 = denoised_complex_128 * y_abs_max
    
    # Map 128 bins back to full 1024-bin STFT
    full_spec = torch.zeros(1, opts.stft_nfft, 32, dtype=torch.complex64, device=opts.device)
    full_spec[:, 0:64, :] = denoised_complex_128[:, 64:, :]
    full_spec[:, -64:, :] = denoised_complex_128[:, :64, :]
    
    # Pad time dimension back to 33 for exact 1024-sample ISTFT synthesis
    full_spec_33 = torch.cat([full_spec, full_spec[:, :, -1:]], dim=-1)
    recon_iq = torch.istft(
        full_spec_33,
        n_fft=opts.stft_nfft,
        hop_length=opts.stft_overlap,
        win_length=opts.stft_window,
        window=window,
        length=len(sig_iq)
    )
    return recon_iq.squeeze(0).cpu().numpy()

# --------------------------------------------------------------------------
# Helper: Standard LoRa FFT Dechirping
# --------------------------------------------------------------------------
def dechirp_estimate_symbol(sig, ref_chirp, sf, fs, bw):
    """Traditional LoRa dechirping + FFT peak detection"""
    dechirped = sig * ref_chirp
    if fs == bw:
        spectrum = np.fft.fft(dechirped)
    else:
        spectrum = np.fft.fftshift(np.fft.fft(dechirped))
    power = np.abs(spectrum) ** 2
    
    fft_len = len(power)
    center = fft_len // 2
    bins = 2 ** sf
    upper_freq = power[center : center + bins]
    lower_freq = power[center - bins : center]
    combine = upper_freq + lower_freq
    return int(np.argmax(combine))


In [524]:
# File path to the LoRaSeek checkpoint (relative to workspace root)
loraseek_checkpoint = 'models/loraseek/checkpoints/sf7/v9_checkpoints/75000_LoRaSeek.pkl'

loraseek, opts = LoRaSeek_Init(loraseek_checkpoint, sf=7)
window = torch.hann_window(opts.stft_window, device=opts.device)


LoRaSeek Configuration:
free_gpu_id: 0
x_image_channel: 2
y_image_channel: 2
conv_kernel_size: 3
conv_padding_size: 1
lstm_dim: 400
fc1_dim: 600
sf: 7
bw: 125000
fs: 1000000
server: False
normalization: True
init_zero_weights: False
train_iters: 150000
load_iters: 100000
batch_size: 8
num_workers: 1
lr: 0.0002
sorting_type: -1
scaling_for_imaging_loss: 128
scaling_for_classification_loss: 1
beta1: 0.5
beta2: 0.999
root_path: ./
evaluations_dir: evaluations
data_dir: /path/to/raw_sf7_cross_instance
network: end2end
groundtruth_code: 35
code_list: [np.float64(0.0), np.float64(0.1), np.float64(0.2), np.float64(0.3), np.float64(0.4), np.float64(0.5), np.float64(0.6), np.float64(0.7), np.float64(0.8), np.float64(0.9), np.float64(1.0), np.float64(1.1), np.float64(1.2), np.float64(1.3), np.float64(1.4), np.float64(1.5), np.float64(1.6), np.float64(1.7), np.float64(1.8), np.float64(1.9), np.float64(2.0), np.float64(2.1), np.float64(2.2), np.float64(2.3), np.float64(2.4), np.float64(2.5), np.fl

In [525]:
from sean_utils.LoRa import LoRa

# 1. Parameters
sf = 7
bw = 125000
fs = 1_000_000
code_word = 39  # symbol value between 0 and 2^sf - 1
snr_db = -19

# 2. Initialize LoRa & references
lora = LoRa(sf=sf, bw=bw)
ref_down = lora.gen_symbol_fs(0, sf=sf, bw=bw, down=True, Fs=fs)
ref_up = lora.gen_symbol_fs(0, sf=sf, bw=bw, down=False, Fs=fs)

# 3. Generate LoRa symbol using gen_symbol_fs
clean_symbol = lora.gen_symbol_fs(code_word=code_word, sf=sf, bw=bw, down=False, Fs=fs)

# 4. Add AWGN noise using add_awgn_noise
noisy_symbol = lora.add_awgn_noise(clean_symbol, snr_db=snr_db)

# 5. Check shapes and verify measured SNR
measured_snr = lora.calculate_snr_db(clean_symbol, noisy_symbol)
print(f"Clean Symbol Shape : {clean_symbol.shape}, dtype: {clean_symbol.dtype}")
print(f"Noisy Symbol Shape : {noisy_symbol.shape}, dtype: {noisy_symbol.dtype}")
print(f"Target SNR         : {snr_db} dB")
print(f"Measured SNR       : {measured_snr:.2f} dB")


Clean Symbol Shape : (1024,), dtype: complex128
Noisy Symbol Shape : (1024,), dtype: complex128
Target SNR         : -19 dB
Measured SNR       : -18.98 dB


In [526]:
# 1. Denoise noisy IQ using LoRaSeek + ISTFT to get clean IQ signal in time-domain
denoised_symbol_iq = denoise_with_loraseek_istft(noisy_symbol, loraseek, opts, window)

# 2. Demodulate using normal dechirping on both Denoised IQ and Raw Noisy IQ
pred_denoised = dechirp_estimate_symbol(denoised_symbol_iq, ref_down, sf, fs, bw)
pred_noisy = dechirp_estimate_symbol(noisy_symbol, ref_down, sf, fs, bw)

print(f"Target Code Word                 : {code_word}")
print(f"Demodulated from Denoised IQ (AI): {pred_denoised} (Match: {pred_denoised == code_word})")
print(f"Demodulated from Raw Noisy IQ    : {pred_noisy} (Match: {pred_noisy == code_word})")


Target Code Word                 : 39
Demodulated from Denoised IQ (AI): 39 (Match: True)
Demodulated from Raw Noisy IQ    : 38 (Match: False)


In [527]:
import numpy as np
from sean_utils.LoRa import LoRa

# --------------------------------------------------------------------------
# 1. Parameters
# --------------------------------------------------------------------------
sf = 7
bw = 125000
fs = 1_000_000
N_sym = int(2**sf * fs / bw)  # 1024 samples per frame/symbol
lora = LoRa(sf=sf, bw=bw)

# Noise floor power (-10 dB)
snr_db = -21
noise_power = 10 ** (snr_db / 10)
noise_std = np.sqrt(noise_power / 2)

# --------------------------------------------------------------------------
# 2. Build 4-frame packet: [Noise Only, Up Chirp, Down Chirp, Noise Only]
# --------------------------------------------------------------------------
# Frame 0: Noise only
frame_noise_1 = np.random.normal(0, noise_std, N_sym) + 1j * np.random.normal(0, noise_std, N_sym)

# Frame 1: Up chirp (code_word = 0, down=False)
up_chirp_clean = lora.gen_symbol_fs(code_word=0, sf=sf, bw=bw, down=False, Fs=fs)
frame_up = lora.awgn_iq(up_chirp_clean, snr_db)

# Frame 2: Down chirp (code_word = 0, down=True)
down_chirp_clean = lora.gen_symbol_fs(code_word=0, sf=sf, bw=bw, down=True, Fs=fs)
frame_down = lora.awgn_iq(down_chirp_clean, snr_db)

# Frame 3: Noise only
frame_noise_2 = np.random.normal(0, noise_std, N_sym) + 1j * np.random.normal(0, noise_std, N_sym)

# Concatenate into full 4-frame packet
packet = np.concatenate([frame_noise_1, frame_up, frame_down, frame_noise_2])
print(f"Base 4-Frame Packet Shape: {packet.shape} (total samples = {len(packet)})")

# --------------------------------------------------------------------------
# 3. Add CFO and Symbol Timing Offset (STO)
# --------------------------------------------------------------------------
cfo_hz = 2000  # Carrier Frequency Offset in Hz (exact 2 chips for SF=7)
sto_samples = 24    # Timing offset in samples (exact 3 chips advance)

# Apply STO: advance (shift to front) instead of delaying to back
# Signal arrives earlier -> drops sto_samples from front and pads noise at the back
sto_tail_noise = np.random.normal(0, noise_std, sto_samples) + 1j * np.random.normal(0, noise_std, sto_samples)
packet_with_sto = np.concatenate([packet[sto_samples:], sto_tail_noise])

# Apply continuous phase CFO across entire packet: exp(j * 2 * pi * CFO * t)
t = np.arange(len(packet_with_sto)) / fs
rx_packet = packet_with_sto * np.exp(1j * 2 * np.pi * cfo_hz * t)

# --------------------------------------------------------------------------
# 4. Extract Up Chirp and Down Chirp frames (nominal receiver window)
# --------------------------------------------------------------------------
# Slicing at nominal frame boundaries captures the STO impairment
up_chirp_impaired = rx_packet[N_sym : 2 * N_sym]
down_chirp_impaired = rx_packet[2 * N_sym : 3 * N_sym]

print(f"\nApplied Impairments:")
print(f"  - CFO : {cfo_hz:.2f} Hz ({cfo_hz * (2**sf) / bw:.2f} chips)")
print(f"  - STO : {sto_samples} samples ({sto_samples / (fs/bw):.2f} chips, shifted to front)")
print(f"\nPrepared Frames ready for use:")
print(f"  - up_chirp_impaired   : shape={up_chirp_impaired.shape}, dtype={up_chirp_impaired.dtype}")
print(f"  - down_chirp_impaired : shape={down_chirp_impaired.shape}, dtype={down_chirp_impaired.dtype}")


Base 4-Frame Packet Shape: (4096,) (total samples = 4096)

Applied Impairments:
  - CFO : 2000.00 Hz (2.05 chips)
  - STO : 24 samples (3.00 chips, shifted to front)

Prepared Frames ready for use:
  - up_chirp_impaired   : shape=(1024,), dtype=complex128
  - down_chirp_impaired : shape=(1024,), dtype=complex128


In [528]:
# --------------------------------------------------------------------------
# 5. Denoise Up-Chirp with LoRaSeek + ISTFT, then Dechirp
# --------------------------------------------------------------------------
# 1. Denoise up-chirp IQ signal
up_denoised_iq = denoise_with_loraseek_istft(up_chirp_impaired, loraseek, opts, window)

# 2. Normal dechirping using reference down-chirp
f_up_ls = dechirp_estimate_symbol(up_denoised_iq, ref_down, sf, fs, bw)

print(f"LoRaSeek (Denoised + Dechirp) Up-Chirp Symbol (f_up) : {f_up_ls}")


LoRaSeek (Denoised + Dechirp) Up-Chirp Symbol (f_up) : 5


In [529]:
# --------------------------------------------------------------------------
# 6. Denoise Down-Chirp with LoRaSeek + ISTFT, then Dechirp
# --------------------------------------------------------------------------
# LoRaSeek was trained on up-chirps; take conjugate -> denoise -> conjugate back
down_denoised_iq = np.conj(denoise_with_loraseek_istft(np.conj(down_chirp_impaired), loraseek, opts, window))

# Normal dechirping using reference up-chirp
f_down_ls = dechirp_estimate_symbol(down_denoised_iq, ref_up, sf, fs, bw)

print(f"LoRaSeek (Denoised + Dechirp) Down-Chirp Symbol (f_down) : {f_down_ls}")


LoRaSeek (Denoised + Dechirp) Down-Chirp Symbol (f_down) : 127


In [530]:
# --------------------------------------------------------------------------
# 7. CFO Integer & STO Integer Estimation using LoRaSeek Denoised Symbols
# (Based on closed-form up/down chirp pairing from recent_works.py)
# --------------------------------------------------------------------------
def to_nearest_N_center(x, n_classes):
    return x - (n_classes // 2) * round(x / (n_classes // 2))

def wrap_symmetric(x, n):
    x = x % n
    if x > n / 2:
        x -= n
    return x

N = opts.n_classes  # 128 for SF=7
over_sample = opts.fs / opts.bw  # 8 samples per chip

# 1. CFO Integer Estimation: (f_up + f_down) / 2
cfo_raw_ls = (f_up_ls + f_down_ls) / 2.0
cfo_int_chips_ls = int(round(to_nearest_N_center(cfo_raw_ls, N)))
cfo_int_hz_ls = (cfo_int_chips_ls * opts.bw) / N

# 2. STO Integer Estimation: f_up - CFO_int
sto_int_chips_ls = int(wrap_symmetric(f_up_ls - cfo_int_chips_ls, N))
sto_int_samples_ls = sto_int_chips_ls * over_sample

print("=" * 60)
print("LoRaSeek-based Synchronization (CFO & STO Estimation):", snr_db)
print("=" * 60)
print(f"Decoded Symbols     : f_up = {f_up_ls}, f_down = {f_down_ls}")
print(f"Estimated CFO (Int) : {cfo_int_chips_ls} chips  -->  {cfo_int_hz_ls:.2f} Hz")
print(f"Estimated STO (Int) : {sto_int_chips_ls} chips  -->  {sto_int_samples_ls:.1f} samples")
print("-" * 60)
print("Ground Truth Comparison:")
print(f"Applied CFO         : {cfo_hz:.2f} Hz ({cfo_hz * N / opts.bw:.2f} chips)")
print(f"Applied STO         : {sto_samples} samples ({sto_samples / over_sample:.2f} chips)")
print("=" * 60)


LoRaSeek-based Synchronization (CFO & STO Estimation): -21
Decoded Symbols     : f_up = 5, f_down = 127
Estimated CFO (Int) : 2 chips  -->  1953.12 Hz
Estimated STO (Int) : 3 chips  -->  24.0 samples
------------------------------------------------------------
Ground Truth Comparison:
Applied CFO         : 2000.00 Hz (2.05 chips)
Applied STO         : 24 samples (3.00 chips)


In [531]:
# --------------------------------------------------------------------------
# 8. Comparison: LoRaSeek AI (Denoised + Dechirp) vs. Raw Noisy Dechirping
# --------------------------------------------------------------------------
# 1. Decode f_up and f_down directly on raw noisy signals using standard dechirping
f_up_raw = dechirp_estimate_symbol(up_chirp_impaired, ref_down, sf, fs, bw)
f_down_raw = dechirp_estimate_symbol(down_chirp_impaired, ref_up, sf, fs, bw)

# 2. Calculate CFO integer and STO integer from raw dechirping
cfo_raw_dec = (f_up_raw + f_down_raw) / 2.0
cfo_int_chips_raw = int(round(to_nearest_N_center(cfo_raw_dec, N)))
cfo_int_hz_raw = (cfo_int_chips_raw * opts.bw) / N

sto_int_chips_raw = int(wrap_symmetric(f_up_raw - cfo_int_chips_raw, N))
sto_int_samples_raw = sto_int_chips_raw * over_sample

# 3. Print side-by-side comparison table
print("=" * 80)
print("COMPARISON: LoRaSeek (Denoised IQ + Dechirp) vs. Raw Noisy Dechirping", snr_db)
print("=" * 80)
print(f"{'Metric':<25} | {'Ground Truth':<15} | {'LoRaSeek AI':<18} | {'Raw Dechirping':<15}")
print("-" * 80)
print(f"{'f_up symbol':<25} | {'-':<15} | {f_up_ls:<18} | {f_up_raw:<15}")
print(f"{'f_down symbol':<25} | {'-':<15} | {f_down_ls:<18} | {f_down_raw:<15}")
print(f"{'CFO (chips)':<25} | {cfo_hz * N / opts.bw:<15.2f} | {cfo_int_chips_ls:<18} | {cfo_int_chips_raw:<15}")
print(f"{'CFO (Hz)':<25} | {cfo_hz:<15.2f} | {cfo_int_hz_ls:<18.2f} | {cfo_int_hz_raw:<15.2f}")
print(f"{'STO (chips)':<25} | {sto_samples / over_sample:<15.2f} | {sto_int_chips_ls:<18} | {sto_int_chips_raw:<15}")
print(f"{'STO (samples)':<25} | {sto_samples:<15.1f} | {sto_int_samples_ls:<18.1f} | {sto_int_samples_raw:<15.1f}")
print("=" * 80)


COMPARISON: LoRaSeek (Denoised IQ + Dechirp) vs. Raw Noisy Dechirping -21
Metric                    | Ground Truth    | LoRaSeek AI        | Raw Dechirping 
--------------------------------------------------------------------------------
f_up symbol               | -               | 5                  | 5              
f_down symbol             | -               | 127                | 80             
CFO (chips)               | 2.05            | 2                  | -22            
CFO (Hz)                  | 2000.00         | 1953.12            | -21484.38      
STO (chips)               | 3.00            | 3                  | 27             
STO (samples)             | 24.0            | 24.0               | 216.0          
